In [ ]:
from cosipy.nonimaging.bgo.bc_tools_localization import BGOLocalizerBCT
import numpy as np


In [ ]:
# Inizializza con i tre LUT (pickle) e nside

dir_path = "/data/test_newrepo/"
run_name="run10"
localizer = BGOLocalizerBCT(
    soft_lut_path=dir_path+'/soft_lut_' + run_name + '.pkl',
    medium_lut_path=dir_path+'/medium_lut_' + run_name + '.pkl',
    hard_lut_path= dir_path+'/hard_lut_' + run_name + '.pkl',
    nside=64,
)

In [ ]:
def ra_dec_to_theta_phi(ra, dec):

    theta = 90-dec
    
    phi = ra
    
    return theta, phi
def spherical_to_radec_deg(theta_deg, phi_deg):

    dec = 90.0 - theta_deg
    ra = phi_deg % 360.0
    return ra, dec


In [ ]:
# Counts order ['BGO_X0','BGO_X1','BGO_Y0','BGO_Y1','BGO_Z0','BGO_Z1']
# Simulated GRB
# Spectrum Band 10 10000 -1.9 -3.7 230
# Flux 14.58 ph/cm2/s
# True position theta=84.021 phi=49.922

true_ra,true_dec = spherical_to_radec_deg(84.021,49.922)
s_counts = [46, 316, 33, 374, 47, 34]

#for testing purpose generate random Poissonian background 
mean_counts = np.array([57.6053, 58.7157, 51.4131, 48.2891, 47.7293, 45.9617])
random_bkg = np.random.poisson(np.array([mean_counts[3],mean_counts[2],mean_counts[5],mean_counts[4],mean_counts[1],mean_counts[0]])*20)

b_counts = np.array([mean_counts[3],mean_counts[2],mean_counts[5],mean_counts[4],mean_counts[1],mean_counts[0]])*20
s_counts = s_counts+random_bkg

In [ ]:
result = localizer.localize(s_counts, b_counts)
print(
    result["label"],
    result["sqrt_ts"],
    result["ra_deg"],
    result["dec_deg"],
    result["eq_radius_deg"],
)

In [ ]:
ra_dec_to_theta_phi(result["ra_deg"],result["dec_deg"])

In [ ]:
# Plot opzionale
%matplotlib inline

from astropy.coordinates import SkyCoord
import astropy.units as u

true_coord = SkyCoord(ra=true_ra * u.deg, dec=true_dec * u.deg, frame="icrs")
localizer.plot(result,true_coord=true_coord, show=True,save_path="/tmp/plot.png")
